# Práctica ML y DL - 01 Preprocesamiento
En este notebook realizaremos el preprocesamiento de los datos proporcionados por la empresa *Se nos fue de las Manos*. 
Tomaremos las mejores ideas de los proyectos de referencia:
- Limpieza de la variable `ALTITUD` (rangos a media) y su imputación por medias de estación.
- Imputación de la variable `SUPERFICIE` utilizando los datos históricos por finca.
- Agregación de las variables meteorológicas de `METEO` y `ETO` para complementar la información de cada campaña.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Cargar los datos
df_train = pd.read_csv('data/TRAIN.txt', sep='|', encoding='utf-8')
print("Dimensiones TRAIN:", df_train.shape)
df_train.head()

Dimensiones TRAIN: (8526, 11)


,CAMPAÑA,ID_FINCA,ID_ZONA,ID_ESTACION,ALTITUD,VARIEDAD,MODO,TIPO,COLOR,SUPERFICIE,PRODUCCION
0,14,76953,515,4,660,26,2,0,1,0.0,22215.0
1,14,84318,515,4,660,26,2,0,1,0.0,22215.0
2,14,85579,340,4,520,32,2,0,1,0.0,20978.0
3,14,69671,340,4,520,32,2,0,1,0.0,40722.0
4,14,14001,852,14,NaN,81,1,0,1,0.0,14126.0


## 1. Limpieza de ALTITUD
La variable `ALTITUD` a veces viene expresada como un rango (ej. "600-700"). Las transformaremos a la media del rango y la convertiremos a numérico.

In [2]:
def curar_altitud(x):
    if pd.isna(x):
        return np.nan
    x_str = str(x).strip()
    if '-' in x_str:
        partes = x_str.split('-')
        return (float(partes[0]) + float(partes[1])) / 2
    return float(x_str)

df_train['ALTITUD'] = df_train['ALTITUD'].apply(curar_altitud)

# Imputar ALTITUD faltante usando la media de su ID_ESTACION
altitud_medias = df_train.groupby('ID_ESTACION')['ALTITUD'].mean()
df_train['ALTITUD'] = df_train.apply(
    lambda row: altitud_medias[row['ID_ESTACION']] if pd.isna(row['ALTITUD']) else row['ALTITUD'], 
    axis=1
)

print("Nulos en ALTITUD:", df_train['ALTITUD'].isna().sum())

Nulos en ALTITUD: 0


## 2. Imputación de SUPERFICIE

La variable `SUPERFICIE` suele faltar en años anteriores a la campaña 2020. Una vez examinados los registros del dataset, la operativa más habitual para recuperar esta información consiste en utilizar como base de verdad las campañas donde sí disponemos de datos consolidados (20, 21 y 22) y propagar dichos valores hacia el pasado. 
El algoritmo aplicado para imputar estas superficies sigue las dos casuísticas principales que hemos encontrado:
1. **Fincas con un único registro anual:** Si una finca solo cuenta con un registro en cada campaña y tenemos su medida de superficie en las campañas recientes (20, 21 y 22), imputaremos directamente el valor de esa superficie al resto de campañas históricas de la misma finca.
2. **Fincas divididas en varios registros (policultivos):** Si la finca está subdividida en varios registros por campaña, validamos primero que las variedades de uva coincidan. De ser así, localizamos esa misma combinación única de "Finca + Variedad" en las campañas recientes (20, 21 y 22) y le imputamos su superficie específica.
Para los pocos registros "huérfanos" (fincas muy antiguas que no sobrevivieron hasta 2020), aplicamos un relleno local hacia atrás/hacia adelante y, en última instancia, la superficie mediana de su variedad de uva correspondiente.

In [3]:
# Reemplazamos 0 por NaN para poder imputarlo si lo hubiera
df_train['SUPERFICIE'] = df_train['SUPERFICIE'].replace(0, np.nan)
# Extraemos el histórico confiable: las superficies conocidas de las campañas 20, 21, 22
df_sup_reciente = df_train[df_train['CAMPAÑA'].isin([20, 21, 22])].dropna(subset=['SUPERFICIE'])
# ==== CASO 1: Una finca tiene un solo registro por campaña ====
# Calculamos la cantidad máxima de registros de cada finca en cualquier campaña
registros_max_finca = df_train.groupby(['ID_FINCA', 'CAMPAÑA']).size().groupby('ID_FINCA').max()
fincas_un_registro = registros_max_finca[registros_max_finca == 1].index
# Filtramos las superficies recientes solo para esas fincas únicas
df_sup_caso1 = df_sup_reciente[df_sup_reciente['ID_FINCA'].isin(fincas_un_registro)]
# Nos quedamos con la última superficie registrada (la más reciente) para esa finca
sup_dict_caso1 = df_sup_caso1.groupby('ID_FINCA')['SUPERFICIE'].last().to_dict()
# ==== CASO 2: Una finca tiene VARIOS registros por campaña ====
fincas_multi_registro = registros_max_finca[registros_max_finca > 1].index
df_sup_caso2 = df_sup_reciente[df_sup_reciente['ID_FINCA'].isin(fincas_multi_registro)]
# Si tienen varios registros cruzamos ID_FINCA y VARIEDAD para que coincida exactamente
sup_dict_caso2 = df_sup_caso2.groupby(['ID_FINCA', 'VARIEDAD'])['SUPERFICIE'].last().to_dict()
# ---- Aplicar el algoritmo ----
def imputar_superficie(row):
    # Si ya tiene una superficie válida, la mantenemos
    if pd.notna(row['SUPERFICIE']):
        return row['SUPERFICIE']
    
    # Aplicar Caso 1 (Conocemos finca única)
    if row['ID_FINCA'] in fincas_un_registro:
        return sup_dict_caso1.get(row['ID_FINCA'], np.nan)
    
    # Aplicar Caso 2 (Finca con múltiples registros -> macheamos Finca + Variedad)
    if row['ID_FINCA'] in fincas_multi_registro:
        return sup_dict_caso2.get((row['ID_FINCA'], row['VARIEDAD']), np.nan)
    
    return np.nan
# Ejecutamos la función fila a fila
df_train['SUPERFICIE'] = df_train.apply(imputar_superficie, axis=1)
# ---- Casos "Huerfanos" (Respaldo final) ----
# Si una finca es de antes de 2020 y fue destruida (no tiene historico en 20,21,22)
# arrastramos el valor más cercano hacia arriba o abajo (bfill/ffill) que tenga dentro de la finca
df_train['SUPERFICIE'] = df_train.groupby('ID_FINCA')['SUPERFICIE'].transform(lambda x: x.bfill().ffill())
# Si aún quedan fincas 100% nulas, imputamos con la media de su variedad
superficie_medias = df_train.groupby('VARIEDAD')['SUPERFICIE'].mean()
df_train['SUPERFICIE'] = df_train.apply(
    lambda row: superficie_medias.get(row['VARIEDAD']) if pd.isna(row['SUPERFICIE']) else row['SUPERFICIE'],
    axis=1
)
# Y en el caso rarísimo de que la variedad en sí entera sea nula, aplicamos mediana global
df_train['SUPERFICIE'] = df_train['SUPERFICIE'].fillna(df_train['SUPERFICIE'].median())
print("Nulos en SUPERFICIE:", df_train['SUPERFICIE'].isna().sum())


Nulos en SUPERFICIE: 0


## 3. Variables temporales / Históricas
Vamos a calcular la producción media histórica de cada finca hasta el año anterior, para darle al modelo contexto sobre si la finca es de alta o baja producción.

In [4]:
df_train.sort_values(by=['ID_FINCA', 'CAMPAÑA'], inplace=True)

# Produccion media histórica (excluyendo la campaña actual para evitar data leakage)
df_train['PRODUCCION_HISTORICA_MEDIA'] = df_train.groupby('ID_FINCA')['PRODUCCION'].transform(
    lambda x: x.expanding().mean().shift(1)
)

# Para los NA (primer año de la finca), usamos la produccion media global hasta ese momento o 0
df_train['PRODUCCION_HISTORICA_MEDIA'].fillna(0, inplace=True)

df_train.head()

,CAMPAÑA,ID_FINCA,ID_ZONA,ID_ESTACION,ALTITUD,VARIEDAD,MODO,TIPO,COLOR,SUPERFICIE,PRODUCCION,PRODUCCION_HISTORICA_MEDIA
671,14,200,86,12,482.5,59,1,0,1,0.37,1900.000,0.000000
1819,15,200,86,12,482.5,59,1,0,1,0.37,778.104,1900.000000
2915,16,200,86,12,482.5,59,1,0,1,0.37,1636.200,1339.052000
3970,17,200,86,12,482.5,59,1,0,1,0.37,829.008,1438.101333
5037,18,200,86,12,482.5,59,1,0,1,0.37,607.212,1285.828000


## 4. Datos Meteorológicos
Vamos a agregar las variables de `METEO` a nivel de campaña y estación. Consideraremos el periodo relevante para la uva (por ejemplo, desde septiembre del año anterior hasta agosto de la campaña). Por simplicidad y rapidez, calcularemos estadísticos globales anuales.

In [5]:
df_meteo = pd.read_csv('data/METEO.txt', sep='|', encoding='utf-8')
df_meteo['validTimeUtc'] = pd.to_datetime(df_meteo['validTimeUtc'], errors='coerce')
df_meteo['YEAR'] = df_meteo['validTimeUtc'].dt.year
df_meteo['MONTH'] = df_meteo['validTimeUtc'].dt.month

# Asignamos la campaña meteorologica: de julio a junio
# Si el mes es >= 7, pertenece a la campaña del año siguiente
df_meteo['CAMPAÑA_METEO'] = np.where(df_meteo['MONTH'] >= 7, df_meteo['YEAR'] + 1, df_meteo['YEAR'])
# Ajustamos las campañas al formato de TRAIN (14, 15, ..., 22)
df_meteo['CAMPAÑA_METEO'] = df_meteo['CAMPAÑA_METEO'] - 2000

# Agregamos por ID_ESTACION y CAMPAÑA_METEO
agregaciones = {
    'temperature': ['mean', 'max', 'min'],
    'precip24Hour': ['mean', 'sum'],
    'relativeHumidity': ['mean']
}

df_meteo_agg = df_meteo.groupby(['ID_ESTACION', 'CAMPAÑA_METEO']).agg(agregaciones).reset_index()
# Aplanar nombres de columnas
df_meteo_agg.columns = ['ID_ESTACION', 'CAMPAÑA_METEO'] +                        [f"{col[0]}_{col[1]}" for col in df_meteo_agg.columns[2:]]

# Hacemos merge con TRAIN
df_train = df_train.merge(
    df_meteo_agg, 
    left_on=['ID_ESTACION', 'CAMPAÑA'], 
    right_on=['ID_ESTACION', 'CAMPAÑA_METEO'], 
    how='left'
)
df_train.drop(columns=['CAMPAÑA_METEO'], inplace=True)

# Imputar posibles NA meteorologicos con medias globales de esa campaña
for col in df_meteo_agg.columns[2:]:
    medias_campana = df_train.groupby('CAMPAÑA')[col].transform('mean')
    df_train[col].fillna(medias_campana, inplace=True)
    # Por si queda alguno suelto
    df_train[col].fillna(df_train[col].mean(), inplace=True)

## 5. Guardado de datos preprocesados
Almacenamos los datos para poder utilizarlos en el notebook de modelado.

In [6]:
df_train.to_csv('data/processed_train_data.csv', index=False)
print("Datos preprocesados guardados en data/processed_train_data.csv")

Datos preprocesados guardados en data/processed_train_data.csv
